# RoMa v2 на соседних кадрах трубы

Цель — посмотреть, как готовый плотный матчер RoMa v2 (без дообучения) сопоставляет соседние кадры съёмки трубы.

Данные: `data/pipes/Данные 07 2026/14_эталон_1_...` (эталон 1, длина 8.566 м, диаметр 177.8 мм).

План:
1. Найти в последовательности кадры, где труба в поле зрения, и выделить полосу трубы.
2. Запустить RoMa v2 на паре соседних кадров: поле соответствий, уверенность, поле сдвигов.
3. Сравнить исходные кадры и кадры с вычтенным неподвижным освещением (flat-field).
4. Проверить согласованность: растёт ли сдвиг с шагом между кадрами.
5. Прогнать по отрезку последовательности.

In [ ]:
import os, glob, time
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())  # python.org Python на macOS не видит системные сертификаты

import numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from romav2 import RoMaV2
from romav2.device import device
print("torch", torch.__version__, "| device:", device)
plt.rcParams["figure.dpi"] = 90

## 1. Данные

In [ ]:
SEQ = "../data/pipes/Данные 07 2026/14_эталон_1_выдержка_1600_gain_24_первая_хорошая"
files = sorted(glob.glob(os.path.join(SEQ, "*.jpg")))
H0, W0 = cv2.imread(files[0], 0).shape
print(len(files), "кадров, размер", (H0, W0))

# средняя яркость по последовательности: где труба в кадре
step = 10
bright = [cv2.imread(f, cv2.IMREAD_REDUCED_GRAYSCALE_8).mean() for f in files[::step]]
plt.figure(figsize=(9, 2.5)); plt.plot(np.arange(0, len(files), step), bright)
plt.xlabel("индекс кадра"); plt.ylabel("средняя яркость"); plt.title("Яркость кадров по последовательности"); plt.show()

In [ ]:
I0 = 1100  # кадр из середины, где труба точно в кадре
def read_rgb(i):
    return cv2.cvtColor(cv2.imread(files[i]), cv2.COLOR_BGR2RGB)

# полоса трубы — строки, где яркость выше фона
prof = cv2.cvtColor(read_rgb(I0), cv2.COLOR_RGB2GRAY).mean(1)
thr = prof[:300].mean() + 0.3 * (prof.max() - prof[:300].mean())
rows = np.where(prof > thr)[0]
Y0, Y1 = int(rows.min()) + 50, int(rows.max())
print("полоса трубы: строки", Y0, "–", Y1)

def stretch(a, p=99.5):
    a = a.astype(np.float32)
    return np.clip(a / np.percentile(a, p) * 255, 0, 255).astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(10, 6))
ax[0].imshow(stretch(read_rgb(I0))); ax[0].axhline(Y0, c="y"); ax[0].axhline(Y1, c="y"); ax[0].set_title(f"кадр {I0} (контраст растянут)")
ax[1].plot(prof, np.arange(H0)); ax[1].invert_yaxis(); ax[1].axhline(Y0, c="y"); ax[1].axhline(Y1, c="y"); ax[1].set_title("профиль яркости по строкам")
plt.show()

## 2. Предобработка

Два варианта входа для модели:
- **raw** — вырезанная полоса трубы с растянутым контрастом (кадры очень тёмные);
- **flat** — то же, но кадр поделён на медиану по многим кадрам. Медиана содержит то, что не движется вместе с трубой: пятна освещения, блики от ламп, неравномерность сенсора. После деления остаётся в основном текстура поверхности.

In [ ]:
flat_idx = range(I0 - 200, I0 + 200, 8)
FLAT = np.median(np.stack([read_rgb(i)[Y0:Y1].astype(np.float32) for i in flat_idx]), 0) + 1.0

def prep(i, mode="raw"):
    a = read_rgb(i)[Y0:Y1].astype(np.float32)
    if mode == "flat":
        a = a / FLAT
    return stretch(a)

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
ax[0].imshow(stretch(FLAT)); ax[0].set_title("медиана (неподвижная составляющая)")
ax[1].imshow(prep(I0, "raw")); ax[1].set_title("raw")
ax[2].imshow(prep(I0, "flat")); ax[2].set_title("flat")
for a in ax: a.axis("off")
plt.show()

## 3. Модель

In [ ]:
t = time.time()
model = RoMaV2()            # веса скачиваются один раз (~1 ГБ) в ~/.cache/torch
model.apply_setting("precise")  # 800x800 грубый уровень + 1280x1280 уточнение, двунаправленно
print(f"загрузка {time.time()-t:.1f} c;", "lr", (model.H_lr, model.W_lr), "hr", (model.H_hr, model.W_hr))

In [ ]:
def flow_from_warp(preds, H, W, key="AB"):
    # warp в нормированных координатах [-1,1] -> поле сдвигов в пикселях исходной полосы
    w = preds[f"warp_{key}"][0].float().cpu().numpy()
    ov = preds[f"overlap_{key}"][0, ..., 0].float().cpu().numpy()
    h, ww = w.shape[:2]
    gx = (np.arange(ww) + 0.5) / ww * 2 - 1
    gy = (np.arange(h) + 0.5) / h * 2 - 1
    dx = (w[..., 0] - gx[None]) * W / 2
    dy = (w[..., 1] - gy[:, None]) * H / 2
    return dx, dy, ov

def std_map(preds, key="AB"):
    P = preds[f"precision_{key}"][0].float()
    return (torch.linalg.det(P) ** (-1 / 4)).cpu().numpy()

def match_pair(i, j, mode="raw"):
    A, B = prep(i, mode), prep(j, mode)
    t = time.time(); preds = model.match(A, B)
    if device.type == "mps": torch.mps.synchronize()
    dt = time.time() - t
    dx, dy, ov = flow_from_warp(preds, *A.shape[:2])
    return dict(A=A, B=B, preds=preds, dx=dx, dy=dy, ov=ov, time=dt)

def summary(r, thr=0.5):
    m = r["ov"] > thr
    if m.sum() == 0:
        return dict(dx=np.nan, dy=np.nan, dx_iqr=np.nan, dy_iqr=np.nan, overlap=r["ov"].mean())
    x, y = r["dx"][m], r["dy"][m]
    iqr = lambda v: np.subtract(*np.percentile(v, [75, 25]))
    return dict(dx=np.median(x), dy=np.median(y), dx_iqr=iqr(x), dy_iqr=iqr(y), overlap=r["ov"].mean())

## 4. Пара соседних кадров

Что показываем:
- **B → A**: кадр B, перенесённый в координаты A по найденному полю. Если соответствия верны, он должен совпасть с A.
- **overlap**: уверенность модели, что пиксель A вообще виден на B.
- **std**: оценка неопределённости положения соответствия (в пикселях сети, меньше — лучше).
- **dx, dy**: поле сдвигов в пикселях исходного кадра.

In [ ]:
def show_pair(r, title=""):
    preds = r["preds"]
    H, W = r["A"].shape[:2]
    h, w = preds["warp_AB"].shape[1:3]
    Bt = torch.from_numpy(cv2.resize(r["B"], (w, h))).permute(2, 0, 1)[None].float().to(device) / 255
    BtoA = F.grid_sample(Bt, preds["warp_AB"].float(), mode="bilinear", align_corners=False)[0].permute(1, 2, 0).cpu().numpy()
    fig, ax = plt.subplots(2, 4, figsize=(17, 8))
    ax[0, 0].imshow(r["A"]); ax[0, 0].set_title("A")
    ax[0, 1].imshow(r["B"]); ax[0, 1].set_title("B")
    ax[0, 2].imshow(BtoA * r["ov"][..., None]); ax[0, 2].set_title("B → A (по warp), × overlap")
    ax[0, 3].imshow(np.abs(cv2.resize(r["A"], (w, h)).astype(float) / 255 - BtoA).mean(-1), cmap="magma", vmax=0.3); ax[0, 3].set_title("|A − (B→A)|")
    im = ax[1, 0].imshow(r["ov"], vmin=0, vmax=1); ax[1, 0].set_title("overlap"); plt.colorbar(im, ax=ax[1, 0], shrink=.7)
    im = ax[1, 1].imshow(std_map(preds), vmin=0, vmax=8, cmap="viridis"); ax[1, 1].set_title("std"); plt.colorbar(im, ax=ax[1, 1], shrink=.7)
    lim = max(10, np.percentile(np.abs(np.r_[r["dx"].ravel(), r["dy"].ravel()]), 95))
    im = ax[1, 2].imshow(r["dx"], cmap="coolwarm", vmin=-lim, vmax=lim); ax[1, 2].set_title("dx, px"); plt.colorbar(im, ax=ax[1, 2], shrink=.7)
    im = ax[1, 3].imshow(r["dy"], cmap="coolwarm", vmin=-lim, vmax=lim); ax[1, 3].set_title("dy, px"); plt.colorbar(im, ax=ax[1, 3], shrink=.7)
    for a in ax.ravel(): a.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

def show_hist(r, title=""):
    m = r["ov"] > 0.5
    fig, ax = plt.subplots(1, 2, figsize=(11, 2.8))
    for a, v, n in zip(ax, [r["dx"], r["dy"]], ["dx", "dy"]):
        a.hist(v[m], bins=200, range=(-300, 300)); a.set_title(f"{title}: {n} (overlap>0.5)"); a.set_yscale("log")
    plt.show()

r_raw = match_pair(I0, I0 + 1, "raw")
print(f"время: {r_raw['time']:.2f} c", summary(r_raw))
show_pair(r_raw, f"raw: кадры {I0} и {I0+1}")
show_hist(r_raw, "raw")

In [ ]:
r_flat = match_pair(I0, I0 + 1, "flat")
print(f"время: {r_flat['time']:.2f} c", summary(r_flat))
show_pair(r_flat, f"flat: кадры {I0} и {I0+1}")
show_hist(r_flat, "flat")

## 5. Проверка согласованности по шагу между кадрами

Истинного сдвига у нас нет. Косвенная проверка: если труба движется равномерно, сдвиг между кадрами *i* и *i+k* должен расти примерно линейно по *k*, а уверенность — плавно падать. Скачки и резкое падение overlap означают ложные соответствия.

In [ ]:
rows = []
for mode in ["raw", "flat"]:
    for k in [1, 2, 3, 5, 10, 20]:
        r = match_pair(I0, I0 + k, mode)
        rows.append(dict(mode=mode, k=k, **summary(r)))
gap = pd.DataFrame(rows).round(2)
gap

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for mode, g in gap.groupby("mode"):
    ax[0].plot(g.k, g.dx, "o-", label=mode); ax[1].plot(g.k, g.dy, "o-", label=mode); ax[2].plot(g.k, g.overlap, "o-", label=mode)
for a, t in zip(ax, ["медиана dx, px", "медиана dy, px", "средний overlap"]):
    a.set_xlabel("шаг k"); a.set_title(t); a.legend(); a.grid(alpha=.3)
plt.show()

## 6. Отрезок последовательности

Соседние пары подряд: медианный сдвиг, разброс и overlap для каждой пары. Если модель уверенно отслеживает поверхность, ряд сдвигов должен быть гладким.

In [ ]:
N = 40
seq = []
for mode in ["raw", "flat"]:
    for i in range(I0, I0 + N):
        r = match_pair(i, i + 1, mode)
        seq.append(dict(mode=mode, i=i, **summary(r)))
seq = pd.DataFrame(seq)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for mode, g in seq.groupby("mode"):
    ax[0].plot(g.i, g.dx, ".-", label=mode); ax[1].plot(g.i, g.dy, ".-", label=mode); ax[2].plot(g.i, g.overlap, ".-", label=mode)
for a, t in zip(ax, ["dx (соседние), px", "dy (соседние), px", "overlap"]):
    a.set_xlabel("кадр i"); a.set_title(t); a.legend(); a.grid(alpha=.3)
plt.show()
seq.groupby("mode")[["dx", "dy", "dx_iqr", "dy_iqr", "overlap"]].describe().T.round(2)

## 7. Независимая проверка: шаблонное сопоставление

Классический метод без нейросети: берём центральную часть кадра A (после flat-field и high-pass) и ищем её в кадре B по нормированной корреляции на всём кадре. Пик корреляции и есть оценка сдвига. Это не истина, но подход совсем другой, поэтому совпадение с RoMa о чём-то говорит.

In [ ]:
s = 4
def small_gray(i):
    a = cv2.cvtColor(read_rgb(i)[Y0:Y1], cv2.COLOR_RGB2GRAY).astype(np.float32)
    return cv2.resize(a, None, fx=1/s, fy=1/s, interpolation=cv2.INTER_AREA)
FLAT_G = np.median(np.stack([small_gray(i) for i in flat_idx]), 0) + 1
def hp(i):
    a = small_gray(i) / FLAT_G; a = a - cv2.GaussianBlur(a, (0, 0), 6)
    return (a - a.mean()) / a.std()

def template_shift(i, j):
    A, B = hp(i), hp(j); H, W = A.shape
    r = cv2.matchTemplate(B, A[H//4:3*H//4, W//4:3*W//4], cv2.TM_CCOEFF_NORMED)
    y, x = np.unravel_index(r.argmax(), r.shape)
    return (x - W//4) * s, (y - H//4) * s, r.max()

tm = pd.DataFrame([dict(i=i, zip_=None, **dict(zip(["dx", "dy", "score"], template_shift(i, i + 1)))) for i in range(I0, I0 + N)]).drop(columns="zip_")
plt.figure(figsize=(6, 4))
sc = plt.scatter(tm.dx, tm.dy, c=tm.score, cmap="viridis", s=40); plt.colorbar(sc, label="score")
plt.xlabel("dx, px"); plt.ylabel("dy, px"); plt.title("Шаблонное сопоставление: сдвиг соседних кадров"); plt.grid(alpha=.3); plt.show()
tm.sort_values("score", ascending=False).head(8).round(3)

Ненулевые ответы шаблонного метода почти одинаковые, около **(55, −484) px за кадр**, и самые уверенные пики (наибольший score) именно там. Так выглядит равномерное движение трубы: −484 по y — вращение, +55 по x — продвижение вдоль оси. Ответы около (0, 0) — артефакт: неподвижная составляющая (свет, рисунок сенсора) убирается делением на медиану не полностью.

Считаем ориентир (55, −484) и смотрим, у какой доли пар RoMa попадает рядом с ним.

In [ ]:
REF = np.array([55.0, -484.0])
def near(df, tol=60):
    return np.hypot(df.dx - REF[0], df.dy - REF[1]) < tol

all_pts = pd.concat([seq.assign(method="RoMa " + seq["mode"]), tm.assign(method="template")])
print(all_pts.groupby("method").apply(lambda g: f"{near(g).mean():.0%} пар рядом с (55, −484)", include_groups=False).to_string())

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for m_, g in all_pts.groupby("method"):
    ax.scatter(g.dx, g.dy, label=m_, s=30, alpha=.7)
ax.scatter(*REF, marker="*", s=300, c="k", label="ориентир")
ax.set_xlabel("медиана dx, px"); ax.set_ylabel("медиана dy, px"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Сдвиг соседних кадров: RoMa vs шаблон"); plt.show()

## 8. Выводы

- RoMa v2 из коробки запускается на Mac (MPS), около 2–3 с на пару в режиме `precise`.
- Поле соответствий выглядит гладким и уверенным (overlap 0.6–0.9, низкий std) **даже когда оно неверное**. Уверенность модели здесь плохой индикатор правильности.
- По соседним парам ответы RoMa делятся на два режима: правильный (~(55, −484) px) и ложный «почти ноль». Ложный режим возникает, потому что в кадре много неподвижного: пятна освещения, блики, полосы сенсора. Модель сопоставляет их, а не поверхность.
- Деление на медиану (flat-field) помогает мало: неподвижная составляющая убирается не полностью.
- На шаге больше 3 кадров перекрытие маленькое, и RoMa даёт хаотичные ответы.
- Это ровно та ошибка, о которой гипотеза НИР: повторяющаяся текстура плюс помехи дают структурно неверное, но уверенное соответствие.

Что пробовать дальше:
1. Лучше убирать неподвижную составляющую: медиана по большему окну, вычитание в лог-пространстве, маска бликов.
2. Ограничить поиск: движение трубы почти постоянное, можно задавать априорный сдвиг или отбрасывать соответствия около нуля.
3. Оценить длину трубы: сумма dx по всей проезжающей трубе × масштаб (мм/px) — сравнить с 8.566 м.
4. Проверить вторую съёмку того же эталона (`15_...`) и данные 04.2026.